# OpenAI Batch Workflow

This notebook covers the full workflow for concept filtering with the OpenAI Batch API:

1. Generate batch request files
2. Upload the JSONL input file
3. Create a batch job
4. Poll until completion
5. Download output and error files
6. Reconcile outputs with the manifest


In [ ]:
from pathlib import Path
import json
import time
import os
import subprocess

from openai import OpenAI


## Configuration

In [ ]:
INPUT_POOL = Path("output/05_all_concepts.txt")
BATCH_DIR = Path("output/openai_batch")
REQUESTS_PATH = BATCH_DIR / "requests.jsonl"
MANIFEST_PATH = BATCH_DIR / "requests_manifest.json"
OUTPUT_JSONL_PATH = BATCH_DIR / "batch_output.jsonl"
ERROR_JSONL_PATH = BATCH_DIR / "batch_errors.jsonl"
MODEL = "gpt-5.4"
BATCH_SIZE = 50
COMPLETION_WINDOW = "24h"

BATCH_DIR.mkdir(parents=True, exist_ok=True)
api_key = 'REPLACE_WITH_API_KEY'
client = OpenAI(api_key=api_key)


## Step 1: Generate Request Files

Run this if `requests.jsonl` and `requests_manifest.json` do not exist yet, or if you want to regenerate them.

In [ ]:
import subprocess

subprocess.run(
    [
        "python",
        "prepare_openai_batch_requests.py",
        "--input",
        str(INPUT_POOL),
        "--output",
        str(BATCH_DIR),
        "--model",
        MODEL,
        "--batch-size",
        str(BATCH_SIZE),
    ],
    check=True,
)

print(REQUESTS_PATH)
print(MANIFEST_PATH)


## Step 2: Inspect Generated Inputs

In [ ]:
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["total_concepts"], manifest["num_requests"]


In [ ]:
with open(REQUESTS_PATH, "r", encoding="utf-8") as f:
    first_line = f.readline()

json.loads(first_line)


## Step 3: Upload the Batch Input File

In [ ]:
input_file = client.files.create(
    file=REQUESTS_PATH.open("rb"),
    purpose="batch",
)

input_file.id


## Step 4: Create the Batch Job

In [ ]:
batch = client.batches.create(
    input_file_id=input_file.id,
    endpoint="/v1/chat/completions",
    completion_window=COMPLETION_WINDOW,
    metadata={"task": "concept_filtering"},
)

batch.id, batch.status


Save the batch id so you can resume later without recreating the job.

In [ ]:
batch_id_path = BATCH_DIR / "batch_id.txt"
batch_id_path.write_text(batch.id + "\n", encoding="utf-8")
batch_id_path


## Step 5: Retrieve an Existing Batch

Use this cell if you already have a saved batch id.

In [ ]:
batch_id = (BATCH_DIR / "batch_id.txt").read_text(encoding="utf-8").strip()
batch = client.batches.retrieve(batch_id)
batch.id, batch.status


## Step 6: Poll Until Completion

In [ ]:
POLL_INTERVAL_SECONDS = 30
FINAL_STATES = {"completed", "failed", "expired", "cancelled"}

while batch.status not in FINAL_STATES:
    print(batch.status, batch.request_counts)
    time.sleep(POLL_INTERVAL_SECONDS)
    batch = client.batches.retrieve(batch.id)

print(batch.status)
print(batch.request_counts)
print("output_file_id:", batch.output_file_id)
print("error_file_id:", batch.error_file_id)


## Step 7: Download the Output File

In [ ]:
if not batch.output_file_id:
    raise ValueError("No output_file_id found on the batch object")

output_text = client.files.content(batch.output_file_id).text
OUTPUT_JSONL_PATH.write_text(output_text, encoding="utf-8")
OUTPUT_JSONL_PATH


In [ ]:
subprocess.run([
    'python', 'parse_openai_batch_results.py',
    '--results', str(OUTPUT_JSONL_PATH),
    '--manifest', str(MANIFEST_PATH),
    '--output', 'output/openai',
], check=True)

## Step 8: Download the Error File If Present

In [ ]:
if batch.error_file_id:
    error_text = client.files.content(batch.error_file_id).text()
    ERROR_JSONL_PATH.write_text(error_text, encoding="utf-8")
    print(ERROR_JSONL_PATH)
else:
    print("No error file")


## Step 9: Parse Batch Output JSONL

In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


output_rows = load_jsonl(OUTPUT_JSONL_PATH)
len(output_rows)


## Step 10: Index Results by `custom_id`

Batch outputs are not guaranteed to be in the same order as the request file. Use `custom_id` to match rows.

In [ ]:
results_by_custom_id = {}

for row in output_rows:
    custom_id = row["custom_id"]
    response_body = row["response"]["body"]
    message = response_body["choices"][0]["message"]
    content = message["content"]
    results_by_custom_id[custom_id] = {
        "raw_row": row,
        "content": content,
    }

len(results_by_custom_id)


## Step 11: Join with the Manifest

In [ ]:
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

joined = []
for item in manifest["requests"]:
    custom_id = item["custom_id"]
    joined.append({
        "custom_id": custom_id,
        "batch_index": item["batch_index"],
        "concepts": item["concepts"],
        "response_text": results_by_custom_id.get(custom_id, {}).get("content"),
    })

joined[0]


## Step 12: Save a Consolidated Result File

In [ ]:
joined_path = BATCH_DIR / "batch_output_joined.json"
with joined_path.open("w", encoding="utf-8") as f:
    json.dump(joined, f, indent=2, ensure_ascii=False)

joined_path
